# Figure 3: Sensitivity Experiment Maps

Sensitivity experiments with height scaled to 80%, 90%, 110%, 120% of CLM Default.

Data source: 5-year averaged (2010-2014) CLM5 output from `G:/Hangkai/CTH_ET project/Final_data/`

In [ ]:
import netCDF4
import os
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import ListedColormap
from matplotlib import ticker

# ============================================================
# Load all data from 5-year averaged Final_data
# ============================================================
file_path = r'G:\Hangkai\CTH_ET project\Final_data'

# Load PFT structure from Final_data
with netCDF4.Dataset(os.path.join(file_path, 'CLM Default.nc'), 'r') as ds:
    pfts1d_itype_veg = np.array(ds.variables['pfts1d_itype_veg'][:])
    pfts1d_ixy       = np.array(ds.variables['pfts1d_ixy'][:])
    pfts1d_jxy       = np.array(ds.variables['pfts1d_jxy'][:])
    pfts1d_wtgcell   = np.array(ds.variables['pfts1d_wtgcell'][:])

if pfts1d_itype_veg.ndim > 1:
    pfts1d_itype_veg = pfts1d_itype_veg[0, :]
    pfts1d_ixy = pfts1d_ixy[0, :]
    pfts1d_jxy = pfts1d_jxy[0, :]
    pfts1d_wtgcell = pfts1d_wtgcell[0, :]

# Load lat/lon
file_path_input = r'H:\CLM_input'
with netCDF4.Dataset(os.path.join(file_path_input, 'mean.nc'), 'r') as ds:
    lat2d = ds.variables['LATIXY'][:]
    lon2d = ds.variables['LONGXY'][:]
lon2d = lon2d - 180

pft_names = {
    "NET Temperate": 1, "NET Boreal": 2, "NDT Boreal": 3,
    "BET Tropical": 4, "BET Temperate": 5,
    "BDT Tropical": 6, "BDT Temperate": 7, "BDT Boreal": 8,
}
unique_pfts = pft_names.keys()

# Load sensitivity scenarios from Final_data
scenarios_EX = ['Default 0.8', 'Default 0.9', 'CLM Default', 'Default 1.1', 'Default 1.2']
variable_names = [
    'FCEV', 'FCTR', 'QVEGE', 'QVEGT', 'FGEV', 'FSH', 'FSH_G', 'FSH_V', 'EFLX_LH_TOT',
    'EFLX_SOIL_GRND', 'FPSN', 'FPSN_WC', 'FPSN_WJ', 'FPSN_WP', 'Qh', 'Qle', 'Q2M',
    'QFLX_EVAP_TOT', 'QFLX_SNOW_GRND', 'QFLX_RAIN_GRND', 'Rnet', 'TV',
    'TSA', 'Z0HV', 'Z0M', 'Z0MV', 'Z0QV'
]

scenario_mapping_EX = {
    'Default 0.8': 'Default 0.8.nc',
    'Default 0.9': 'Default 0.9.nc',
    'CLM Default': 'CLM Default.nc',
    'Default 1.1': 'Default 1.1.nc',
    'Default 1.2': 'Default 1.2.nc',
}

grid_shape = (len(lat2d), len(lon2d[0]))

data_EX = {}
for scenario in scenario_mapping_EX:
    data_EX[scenario] = {}
    nc_file = os.path.join(file_path, scenario_mapping_EX[scenario])
    with netCDF4.Dataset(nc_file, 'r') as ds:
        for variable in variable_names:
            data_EX[scenario][variable] = ds.variables[variable][:]

# Compute 2D grids
annual_data_EX = {var: {} for var in variable_names}

for scenario in scenario_mapping_EX:
    for variable in variable_names:
        variable_2d_grid = np.zeros(grid_shape)
        for pft_name in unique_pfts:
            pft = pft_names[pft_name]
            pft_indexes = np.where(pfts1d_itype_veg == pft)[0]
            variable_values = np.mean(data_EX[scenario][variable], axis=0) * pfts1d_wtgcell
            variable_pft = variable_values[pft_indexes]

            for i, index in enumerate(pft_indexes):
                row = pfts1d_jxy[index] - 1
                col = pfts1d_ixy[index] - 1
                variable_2d_grid[row, col] += variable_pft[i]

        annual_data_EX[variable][scenario] = variable_2d_grid

print('Data loaded and processed.')

In [ ]:
import cartopy.crs as ccrs

# ============================================================
# Figure 3: Sensitivity experiment maps
# ============================================================
projection = ccrs.PlateCarree(central_longitude=180)
variables = ['ET', 'FCEV', 'FCTR', 'FGEV']

turbo_cmap = plt.get_cmap('coolwarm')
colors_cm = turbo_cmap(np.linspace(0, 1, 200))
colors_cm[95:105] = (1, 1, 1, 1)
white_center_cmap = ListedColormap(colors_cm)

fig, axes = plt.subplots(4, 5, figsize=(25, 18),
                         subplot_kw=dict(projection=projection), dpi=400)

for j, variable in enumerate(variables):
    for i, scenario in enumerate(scenario_mapping_EX):
        ax = axes[j, i]

        if variable == 'ET':
            data_fcev = annual_data_EX['FCEV'][scenario]
            data_fctr = annual_data_EX['FCTR'][scenario]
            data_fgev = annual_data_EX['FGEV'][scenario]
            data_et = data_fcev + data_fctr + data_fgev
        else:
            data_et = annual_data_EX[variable][scenario]

        ylabel_map = {'ET': 'ET', 'FCEV': 'Canopy EV', 'FCTR': 'Canopy TR', 'FGEV': 'Ground EV'}
        ylabel = ylabel_map[variable]

        if scenario == 'CLM Default':
            base_data = data_et.copy()
            base_data[data_et == 0] = np.nan
            plot_data = np.ma.masked_invalid(base_data * 1.0562)  # W/m2 to mm/month
            title = f'CLM Default (mm/month)'
            levels = np.linspace(np.nanmin(plot_data), np.nanmax(plot_data), num=21)
            max_val = np.nanmax(plot_data)
            min_val = np.nanmin(plot_data)
            middle = 0.5 * (min_val + max_val)
            cbar_ticks = [min_val, 0.5*(middle+min_val), middle, 0.5*(middle+max_val), max_val]
            colormap = plt.get_cmap('afmhot_r')
        else:
            if variable == 'ET':
                base_fcev = annual_data_EX['FCEV']['CLM Default']
                base_fctr = annual_data_EX['FCTR']['CLM Default']
                base_fgev = annual_data_EX['FGEV']['CLM Default']
                base_data = base_fcev + base_fctr + base_fgev
            else:
                base_data = annual_data_EX[variable]['CLM Default']
            base_data[base_data == 0] = np.nan
            plot_data = (data_et - base_data) / base_data * 100
            plot_data = np.ma.masked_invalid(plot_data)
            title = f'{scenario} vs Default (%)'
            levels = np.linspace(-10, 10, num=21)
            colormap = white_center_cmap
            cbar_ticks = [-10, -5, 0, 5, 10]

        cs = ax.contourf(lon2d, lat2d, plot_data, levels=levels, cmap=colormap,
                         extend='both', transform=projection)
        ax.coastlines()

        gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False,
                          linewidth=0.5, color='gray', alpha=0.5,
                          xlocs=[-120, 0, 120], ylocs=[-90, -45, 0, 45, 90])
        labels = gl.xlabel_style; labels['size'] = 0; gl.xlabel_style = labels
        labels = gl.ylabel_style; labels['size'] = 0; gl.ylabel_style = labels
        gl.bottom_labels = False; gl.right_labels = False

        cbar = plt.colorbar(cs, ax=ax, orientation='horizontal', pad=0.005,
                            shrink=0.6, ticks=cbar_ticks)
        cbar.formatter = ticker.FormatStrFormatter('%.d')
        cbar.ax.tick_params(labelsize=18)

        if j == 0:
            labels = gl.xlabel_style; labels['size'] = 15; gl.xlabel_style = labels
            ax.set_title(title, fontsize=20)
            cbar.ax.tick_params(labelsize=18)

        if i == 0:
            labels = gl.ylabel_style; labels['size'] = 15; gl.ylabel_style = labels
            ax.annotate(ylabel, xy=(-0.23, 0.5), xycoords='axes fraction',
                        fontsize=25, rotation=90, va='center')

plt.subplots_adjust(hspace=-0.65, wspace=0.03)
plt.savefig('figures/Figure_3_Sensitivity_Maps.png', dpi=300, bbox_inches='tight')
plt.savefig('figures/Figure_3_Sensitivity_Maps.tiff', dpi=300, bbox_inches='tight')
plt.show()
print('Figure 3 saved.')